In [17]:
import xarray as xr
import autoroot
from satpy.scene import Scene
from pyhdf.SD import SD, SDC 
import pandas as pd
import cartopy.crs as ccrs
from pyproj import CRS, Transformer
import rioxarray
import numpy as np
import matplotlib.pyplot as plt
from pyresample import create_area_def
import os
from rs_tools._src.geoprocessing.match import match_timestamps_af
from rs_tools._src.utils.io import get_list_filenames
from pathlib import Path
from utils_msg_modis_aligment import parse_af_dates_from_file, reproject_msg_image, clip_and_align, check_overlap

In [18]:
modis_cm_path =  '/mnt/data8tb/fire_detection/modis/CM'
files = [os.path.join(modis_cm_path, i) for i in os.listdir(modis_cm_path) if i.endswith('.hdf')]
print(f"Found {len(files)} MODIS files")

outpath = '/mnt/data8tb/fire_detection/modis/msg_cm_dataset'

Found 4716 MODIS files


In [19]:
modis_cm_path = '/mnt/data8tb/fire_detection/modis/CM'
seviri_path = "/mnt/seviri/geoprocessed"

#/mnt/outputs/geoprocessed

In [20]:
cm_files = [ i for i in os.listdir(modis_cm_path) if i.endswith('.tif')]
cm_df = pd.DataFrame({'name': cm_files})

In [21]:
cm_df

,name
0,MYD35_L2.A2024167.1405.061.2024168152141.tif
1,MYD35_L2.A2022263.1220.061.2022264194231.tif
2,MYD35_L2.A2020195.1015.061.2020196155024.tif
3,MYD35_L2.A2020187.1100.061.2020188144050.tif
4,MYD35_L2.A2021190.1150.061.2021191152816.tif
...,...
4711,MYD35_L2.A2020214.1225.061.2020215151145.tif
4712,MYD35_L2.A2024154.1005.061.2024156170958.tif
4713,MYD35_L2.A2021262.0920.061.2021262192707.tif
4714,MYD35_L2.A2021186.0855.061.2021186194104.tif


In [22]:
pat = r'\.A(?P<y>\d{4})(?P<doy>\d{3})\.(?P<hh>\d{2})(?P<mm>\d{2})'
parts = cm_df['name'].str.extract(pat)
# to numeric
parts = parts.apply(pd.to_numeric, errors='coerce')
cm_df['datetime'] = (
    pd.to_datetime(parts['y'], format='%Y')
    + pd.to_timedelta(parts['doy'] - 1, unit='D')
    + pd.to_timedelta(parts['hh'], unit='h')
    + pd.to_timedelta(parts['mm'], unit='m')
)
cm_df['utimes'] = cm_df['datetime'].dt.strftime('%Y%m%d%H%M00')
unique_times_cm = cm_df['datetime'].dt.strftime('%Y%m%d%H%M00').unique().tolist()

In [23]:
cm_df

,name,datetime,utimes
0,MYD35_L2.A2024167.1405.061.2024168152141.tif,2024-06-15 14:05:00,20240615140500
1,MYD35_L2.A2022263.1220.061.2022264194231.tif,2022-09-20 12:20:00,20220920122000
2,MYD35_L2.A2020195.1015.061.2020196155024.tif,2020-07-13 10:15:00,20200713101500
3,MYD35_L2.A2020187.1100.061.2020188144050.tif,2020-07-05 11:00:00,20200705110000
4,MYD35_L2.A2021190.1150.061.2021191152816.tif,2021-07-09 11:50:00,20210709115000
...,...,...,...
4711,MYD35_L2.A2020214.1225.061.2020215151145.tif,2020-08-01 12:25:00,20200801122500
4712,MYD35_L2.A2024154.1005.061.2024156170958.tif,2024-06-02 10:05:00,20240602100500
4713,MYD35_L2.A2021262.0920.061.2021262192707.tif,2021-09-19 09:20:00,20210919092000
4714,MYD35_L2.A2021186.0855.061.2021186194104.tif,2021-07-05 08:55:00,20210705085500


In [24]:
files_msg = get_list_filenames(seviri_path, ".nc")

In [25]:
unique_times_msg = list(set(map(parse_af_dates_from_file, files_msg)))

In [26]:
df_matches = match_timestamps_af(unique_times_cm, unique_times_msg, cutoff=15)
df_matches.columns = ['timestamp_cm', 'timestamp_msg']

No valid af mask found for 2024-06-15 14:05:00
No matching af mask found for 2022-09-20 12:20:00
No matching af mask found for 2020-07-13 10:15:00
No matching af mask found for 2020-07-05 11:00:00
No matching af mask found for 2021-07-09 11:50:00
No matching af mask found for 2023-09-11 09:25:00
No matching af mask found for 2021-06-04 09:35:00
No matching af mask found for 2021-08-25 14:20:00
No matching af mask found for 2020-09-23 14:25:00
No matching af mask found for 2020-09-14 11:10:00
No matching af mask found for 2022-07-23 14:30:00
No matching af mask found for 2021-07-27 11:40:00
No matching af mask found for 2022-06-07 11:05:00
No matching af mask found for 2022-09-25 12:30:00
No matching af mask found for 2023-07-06 10:00:00
No matching af mask found for 2022-07-01 13:35:00
No matching af mask found for 2022-07-05 09:45:00
No valid af mask found for 2024-07-15 13:05:00
No matching af mask found for 2022-09-27 14:00:00
No matching af mask found for 2021-09-25 10:20:00
No mat

In [27]:
df_matches

,timestamp_cm,timestamp_msg
0,20210930090500,20210930092743
1,20210930090000,20210930092743
2,20230914144500,20230914151241
3,20210618131000,20210618135744
4,20210907085500,20210907095742
5,20230915135000,20230915145742
6,20210622142500,20210622161242
7,20210601123000,20210601125743
8,20210907090000,20210907095742
9,20210828131500,20210828144242


In [28]:
print(len(df_matches))

40


In [29]:
ref_nat_file = '/mnt/outputs/L1b/MSG4-SEVI-MSG15-0100-NA-20210601232742.451000000Z-NA.nat'
scn = Scene(reader="seviri_l1b_native", filenames=[ref_nat_file])
datasets = scn.available_dataset_names()
scn.load(datasets[1:], generate=False)
dataset = scn['IR_016']
crs_wkt = dataset.attrs['area'].crs_wkt

In [30]:
def count_nans(da, extra_value = None):
    nans_da = da.isnull().sum().item()
    N = da.size
    nan_pct = float(nans_da/N * 100)
    print(f"NaNs: {nan_pct:.2f}%")
    if extra_value:
        values_da =  np.count_nonzero(da.values == -1) 
        final = nans_da + values_da
        nan_pct = float(final/N * 100)
        print(f"NaNs: {nan_pct:.2f}%")
    return nan_pct

In [31]:
for index, row in df_matches.iterrows():
    save_path = os.path.join(outpath, row['timestamp_msg']+'_msg_modis.nc')
    msg_im = xr.open_dataset(os.path.join(seviri_path, '20200929121242_msg.nc'))
    modis_im_name = cm_df[cm_df.utimes == row['timestamp_cm']].name.values[0]
    modis_im = rioxarray.open_rasterio(os.path.join(modis_cm_path, modis_im_name))
    msg_reprojected = reproject_msg_image(msg_im, crs_wkt)
    if check_overlap(modis_im, msg_reprojected):
        msg_clipped, modis_clipped = clip_and_align(msg_reprojected, modis_im)
    else:
        continue
    nans_msg = count_nans(msg_clipped)
    nans_modis = count_nans(modis_clipped, -1)
    if nans_msg<90 and nans_modis<90:
        modis_clipped = modis_clipped.assign_coords(band=["modis_cm"])
        both = xr.concat([msg_clipped, modis_clipped], dim="band")
        both.attrs['modis_time'] = row['timestamp_cm']
        both.attrs['modis_image'] = cm_df[cm_df.utimes == row['timestamp_cm']].name.values[0]
        both.to_netcdf(save_path)
    else:
        continue
    

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/seviri/geoprocessed/20200929121242_msg.nc'